# ASG Airlines - Bronze Layer Ingestion

## Objective

This notebook implements the Bronze layer of the ETL pipeline.

The workflow performs:

- Raw Excel ingestion
- Schema validation
- Logging
- Bronze Parquet generation
- Audit report creation

No business transformations are performed in this layer.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime
import logging

Paths 

In [2]:
# Project Paths

RAW_PATH = Path("../data/raw/UseCase - Airlines.xlsx")
BRONZE_PATH = Path("../data/bronze")
LOG_PATH = Path("../logs")

# Create folders if missing
BRONZE_PATH.mkdir(parents=True, exist_ok=True)
LOG_PATH.mkdir(parents=True, exist_ok=True)

Configure Logging 

In [3]:


log_file = LOG_PATH / "pipeline.log"

logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True
)

logging.info("Bronze Layer Ingestion Started")

Schema Enforcement 

In [4]:
expected_schema = {
    "flights": [
        "flight_id","airline","source","destination",
        "departure_time","arrival_time","duration"
    ],

    "passengers": [
        "passenger_id","first_name","last_name","age",
        "gender","email","phone","aadhaar_id","date_of_birth"
    ],

    "bookings": [
        "booking_id","passenger_id","flight_id","booking_date",
        "status","passport_number","seat_number",
        "emergency_contact_name","emergency_contact_phone"
    ],

    "payments": [
        "payment_id","booking_id","amount","payment_method"
    ]
}

Read excel using pandas 

In [5]:
excel = pd.ExcelFile(RAW_PATH)

bronze = {}

for sheet in excel.sheet_names:

    df = pd.read_excel(excel, sheet_name=sheet)

    bronze[sheet.lower()] = df

    logging.info(f"{sheet} loaded successfully")

print("Sheets Loaded:")
print(list(bronze.keys()))

Sheets Loaded:
['flights', 'payments', 'bookings', 'passengers']


Schema validation

In [6]:
validation_report = []

for table, df in bronze.items():

    expected = set(expected_schema[table])
    actual = set(df.columns)

    missing_cols = expected - actual
    extra_cols = actual - expected

    validation_report.append({
        "Table": table,
        "Status": "PASS" if len(missing_cols)==0 else "FAIL",
        "Missing Columns": ", ".join(missing_cols) if missing_cols else "-",
        "Extra Columns": ", ".join(extra_cols) if extra_cols else "-"
    })

validation_df = pd.DataFrame(validation_report)

validation_df

,Table,Status,Missing Columns,Extra Columns
0,flights,PASS,-,-
1,payments,PASS,-,-
2,bookings,PASS,-,-
3,passengers,PASS,-,-


Parquet Generation


In [11]:
# ==========================================
# Bronze Layer - Datatype Enforcement
# ==========================================

# ---------- Flights ----------
bronze["flights"]["departure_time"] = pd.to_datetime(
    bronze["flights"]["departure_time"],
    errors="coerce"
)

bronze["flights"]["arrival_time"] = pd.to_datetime(
    bronze["flights"]["arrival_time"],
    errors="coerce"
)

# Preserve raw duration as string
bronze["flights"]["duration"] = (
    bronze["flights"]["duration"]
    .astype(str)
    .str.strip()
)

# ---------- Bookings ----------
bronze["bookings"]["booking_date"] = pd.to_datetime(
    bronze["bookings"]["booking_date"],
    errors="coerce"
)

# ---------- Passengers ----------
bronze["passengers"]["date_of_birth"] = pd.to_datetime(
    bronze["passengers"]["date_of_birth"],
    errors="coerce"
)

# ---------- Payments ----------
bronze["payments"]["amount"] = pd.to_numeric(
    bronze["payments"]["amount"],
    errors="coerce"
)

print("Datatypes enforced successfully!")

# Verify datatypes
for table, df in bronze.items():
    print("\n" + "="*45)
    print(table.upper())
    print("="*45)
    print(df.dtypes)

Datatypes enforced successfully!

FLIGHTS
flight_id                 object
airline                   object
source                    object
destination               object
departure_time    datetime64[ns]
arrival_time      datetime64[ns]
duration                  object
dtype: object

PAYMENTS
payment_id         object
booking_id         object
amount            float64
payment_method     object
dtype: object

BOOKINGS
booking_id                         object
passenger_id                       object
flight_id                          object
booking_date               datetime64[ns]
status                             object
passport_number                    object
seat_number                        object
emergency_contact_name             object
emergency_contact_phone            object
dtype: object

PASSENGERS
passenger_id             object
first_name               object
last_name                object
age                       int64
gender                   object
email      

Saving Parquet File
 

In [13]:
# ==========================================
# Save Bronze Layer as Parquet
# ==========================================

for table, df in bronze.items():

    output_path = BRONZE_PATH / f"{table}.parquet"

    df.to_parquet(
        output_path,
        engine="pyarrow",
        index=False
    )

    logging.info(f"{table}.parquet created successfully")

print("Bronze Layer Created Successfully!")

Bronze Layer Created Successfully!


Generating Bronze Audit Report 

In [14]:


from datetime import datetime

audit_report = []

for table, df in bronze.items():

    audit_report.append({
        "table_name": table,
        "rows_loaded": df.shape[0],
        "columns": df.shape[1],
        "missing_values": int(df.isna().sum().sum()),
        "load_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "status": "SUCCESS"
    })

audit_df = pd.DataFrame(audit_report)

audit_df.to_csv(
    BRONZE_PATH / "bronze_ingestion_audit.csv",
    index=False
)

audit_df

,table_name,rows_loaded,columns,missing_values,load_timestamp,status
0,flights,1020,7,41,2026-09-10 16:06:25,SUCCESS
1,payments,1000,4,78,2026-09-10 16:06:25,SUCCESS
2,bookings,1000,9,45,2026-09-10 16:06:25,SUCCESS
3,passengers,1039,9,10,2026-09-10 16:06:25,SUCCESS


# Bronze Layer Summary

## Objective Achieved

The raw ASG Airlines workbook has been successfully ingested into the Bronze layer.

### Deliverables Produced

- Dynamic ingestion of all Excel sheets
- Schema validation
- Datatype enforcement
- Bronze Parquet datasets
- Pipeline logging
- Ingestion audit report

### Bronze Philosophy

No business transformations were applied in this layer. The datasets preserve the original source values and will serve as the input for the Silver Layer, where data cleaning, duplicate resolution, PII masking, and business rule implementation will be performed.